# Developer Salary Analysis: Job Mode & Education Level

**Dataset:** Stack Overflow Annual Developer Survey 2024 — 65,000+ respondents, 112 features. Cleaned subset with non-null salaries (~8,700 records).

**Objective:** Determine whether remote/in-person job mode and education level have statistically significant effects on developer compensation.

**Methods:** Descriptive statistics · Two-sample t-test · One-way ANOVA · Bootstrap resampling (10,000 samples) · Normality and variance assumption checks

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
#Visualize the data
df = pd.read_csv('data/clean_kaggle_data_2024.csv')

# Display first few rows
print(df.head())

## 1. Exploratory Data Analysis

Visualisations from the Stack Overflow 2024 Developer Survey (65,000+ respondents, 112 features). Key dimensions explored: salary by education, work mode distribution, age distribution, job satisfaction.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot salary based on education level
plt.figure(figsize=(10, 6))
sns.boxplot(x='EdLevel', y='ConvertedCompYearly', data=df)
plt.title('Education Level vs Salary')


# Increase the font size of the tick labels
plt.xticks(rotation=90, fontsize=12)
# plt.yticks(fontsize=12)

plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(x='RemoteWork', data=df)
plt.title('Distribution of Remote vs In-Person Work')

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(x='Age', data=df)
plt.title('Age Distribution')
plt.xticks(rotation=90)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Selecting only numerical columns
numerical_cols = df.select_dtypes(include='number')

# Create a correlation matrix
corr_matrix = numerical_cols.corr()

# Plot a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create the plot
plt.figure(figsize=(10, 6))
sns.barplot(x='JobSatPoints_1', y='ConvertedCompYearly', data=df)

# Rotate x-axis labels by 90 degrees
plt.xticks(rotation=90)

# Set titles and labels
plt.title('Job Satisfaction vs Salary')
plt.xlabel('Job Satisfaction (Points)')
plt.ylabel('Converted Compensation (Yearly)')

# Show the plot
plt.show()



## 2. Remote vs In-Person Salary Comparison

Two-group salary analysis: remote workers (n=4,852) vs in-person workers (n=1,845). Outliers removed via IQR method.

### 2.1 Descriptive Statistics

In [ ]:
import numpy as np

# Function to remove outliers using IQR and count them
def remove_outliers(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data < lower_bound) | (data > upper_bound)]  # Identifying outliers
    cleaned_data = data[(data >= lower_bound) & (data <= upper_bound)]
    return cleaned_data, len(outliers)  # Return cleaned data and count of outliers

# Count missing data points for remote and in-person salaries
missing_remote = df[df['RemoteWork'] == 'Remote']['ConvertedCompYearly'].isna().sum()
missing_inperson = df[df['RemoteWork'] == 'In-person']['ConvertedCompYearly'].isna().sum()

# Remove missing data and count outliers for remote salaries
remote_salaries = df[df['RemoteWork'] == 'Remote']['ConvertedCompYearly'].dropna()
remote_salaries_cleaned, remote_outliers = remove_outliers(remote_salaries)

# Remove missing data and count outliers for in-person salaries
inperson_salaries = df[df['RemoteWork'] == 'In-person']['ConvertedCompYearly'].dropna()
inperson_salaries_cleaned, inperson_outliers = remove_outliers(inperson_salaries)

# Output the number of missing data points
print(f"Missing data points in Remote Work Salaries: {missing_remote}")
print(f"Missing data points in In-Person Work Salaries: {missing_inperson}")

# Output the number of outliers removed
print(f"Outliers removed from Remote Work Salaries: {remote_outliers}")
print(f"Outliers removed from In-Person Work Salaries: {inperson_outliers}")

# Descriptive statistics after cleaning
print('\nRemote Work Salaries (Cleaned):')
print(remote_salaries_cleaned.describe())

print('\nIn-Person Work Salaries (Cleaned):')
print(inperson_salaries_cleaned.describe())


### 2.2 Two-Sample T-Test

Manual calculation verified against `scipy.stats.ttest_ind`. T-statistic ≈ 10.055, p < 0.001 — remote workers earn significantly more ($83k vs $43k mean). Note: normality and equal-variance assumptions are violated; bootstrap results provide a more robust comparison.

In [ ]:
import numpy as np

# Extract the salaries for remote and in-person job modes
remote_salaries = df[df['RemoteWork'] == 'Remote']['ConvertedCompYearly'].dropna()
inperson_salaries = df[df['RemoteWork'] == 'In-person']['ConvertedCompYearly'].dropna()

# Compute the sample sizes
n_remote = len(remote_salaries)
n_inperson = len(inperson_salaries)

# Compute the means
mean_remote = np.mean(remote_salaries)
mean_inperson = np.mean(inperson_salaries)

# Compute the variances
var_remote = np.var(remote_salaries, ddof=1)
var_inperson = np.var(inperson_salaries, ddof=1)

# Compute the pooled standard deviation
pooled_std = np.sqrt(((n_remote - 1) * var_remote + (n_inperson - 1) * var_inperson) / (n_remote + n_inperson - 2))

# Compute the t-statistic
t_stat = (mean_remote - mean_inperson) / (pooled_std * np.sqrt(1/n_remote + 1/n_inperson))

# Compute the degrees of freedom
df = n_remote + n_inperson - 2

# Display results
print(f"Mean Remote Salary: {mean_remote}")
print(f"Mean In-Person Salary: {mean_inperson}")
print(f"Pooled Standard Deviation: {pooled_std}")
print(f"T-statistic: {t_stat}")
print(f"Degrees of Freedom: {df}")


In [ ]:
from scipy import stats

# Perform the t-test using scipy's built-in function
t_stat_scipy, p_value = stats.ttest_ind(remote_salaries, inperson_salaries)

print(f"T-statistic (Scipy): {t_stat_scipy}")
print(f"P-value: {p_value}")

### 2.3 Assumption Checks

Shapiro-Wilk (normality) and Levene's (equal variance) tests both return p < 0.05 — assumptions are violated. Results should be interpreted with caution; bootstrap provides the more reliable comparison.

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats

# Q-Q plot for remote salaries
stats.probplot(remote_salaries, dist="norm", plot=plt)
plt.title('Q-Q Plot for Remote Salaries')
plt.show()

# Q-Q plot for in-person salaries
stats.probplot(inperson_salaries, dist="norm", plot=plt)
plt.title('Q-Q Plot for In-Person Salaries')
plt.show()

# Shapiro-Wilk test for normality
shapiro_remote = stats.shapiro(remote_salaries)
shapiro_inperson = stats.shapiro(inperson_salaries)

print(f"Shapiro-Wilk test for Remote Salaries: {shapiro_remote}")
print(f"Shapiro-Wilk test for In-Person Salaries: {shapiro_inperson}")


In [ ]:
# Levene's test for homogeneity of variances
levene_stat, levene_p = stats.levene(remote_salaries, inperson_salaries)
print(f"Levene's Test: Statistic={levene_stat}, P-value={levene_p}")


In [ ]:
# Boxplot for detecting outliers
plt.figure(figsize=(10, 6))
plt.boxplot([remote_salaries, inperson_salaries], labels=['Remote', 'In-Person'])
plt.title('Boxplot of Salaries (Remote vs In-Person)')
plt.show()

### 2.4 Bootstrap Analysis

10,000 resamples. Bootstrapped mean salary difference: ~$35,000 (remote − in-person), stable and consistent with the t-test result.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Number of bootstrap replications
n_bootstrap = 10000

# Function to perform bootstrap sampling
def bootstrap(data, n_samples=n_bootstrap):
    return np.random.choice(data, size=(n_samples, len(data)), replace=True)

# Bootstrap samples for remote and in-person salaries
remote_bootstrap = bootstrap(remote_salaries)
inperson_bootstrap = bootstrap(inperson_salaries)

# Calculate the means of the bootstrap samples
remote_bootstrap_means = np.mean(remote_bootstrap, axis=1)
inperson_bootstrap_means = np.mean(inperson_bootstrap, axis=1)

# Calculate the difference in means for each bootstrap sample
mean_diff_bootstrap = remote_bootstrap_means - inperson_bootstrap_means

# 1. Plot the bootstrapped distributions of salaries for remote and in-person workers
plt.figure(figsize=(10, 6))
plt.hist(remote_bootstrap_means, bins=50, alpha=0.7, label='Remote Work', color='blue')
plt.hist(inperson_bootstrap_means, bins=50, alpha=0.7, label='In-Person Work', color='green')
plt.title('Bootstrapped Distributions of Salaries (Remote vs In-Person)')
plt.xlabel('Salary')
plt.ylabel('Frequency')
plt.legend()
plt.show()

# 2. Plot the distribution of the difference in means
plt.figure(figsize=(10, 6))
plt.hist(mean_diff_bootstrap, bins=50, color='purple', alpha=0.7)
plt.title('Distribution of the Difference in Mean Salaries (Remote - In-Person)')
plt.xlabel('Difference in Salary')
plt.ylabel('Frequency')
plt.show()

# 3. Normalize the differences in means (z-scores) and plot the normalized distribution
mean_diff_mean = np.mean(mean_diff_bootstrap)
mean_diff_std = np.std(mean_diff_bootstrap)
normalized_diff = (mean_diff_bootstrap - mean_diff_mean) / mean_diff_std

plt.figure(figsize=(10, 6))
plt.hist(normalized_diff, bins=50, color='orange', alpha=0.7)
plt.title('Normalized Distribution of the Difference in Mean Salaries')
plt.xlabel('Z-score of the Difference in Salary')
plt.ylabel('Frequency')
plt.show()


### 2.5 Bootstrap T-Test

T-test applied to bootstrapped means confirms the salary gap is statistically significant and robust to the violated parametric assumptions.

In [ ]:
from scipy import stats

# Perform a two-sample t-test on the bootstrapped means
t_stat_bootstrap, p_value_bootstrap = stats.ttest_ind(remote_bootstrap_means, inperson_bootstrap_means)

# Output the t-statistic and p-value from the bootstrapped data
print(f"Bootstrap T-statistic: {t_stat_bootstrap}")
print(f"Bootstrap P-value: {p_value_bootstrap}")

# Comparison with Part 2.b results
print("\nComparison with Part 2.b:")
print(f"T-statistic from Part 2.b: 10.054974198771534")
print(f"Bootstrap T-statistic: {t_stat_bootstrap}")
print(f"P-value from Part 2.b: 1.2672273605881435e-23")
print(f"Bootstrap P-value: {p_value_bootstrap}")

## 3. Education Level & Salary (ANOVA)

Three-group comparison: Bachelor's, Master's, and Professional degree holders. One-way ANOVA with bootstrap validation.

In [ ]:
df = pd.read_csv('data/clean_kaggle_data_2024.csv')

# Extract salaries based on education levels
bachelor_salaries = df[df['EdLevel'] == "Bachelor’s degree"]['ConvertedCompYearly']
master_salaries = df[df['EdLevel'] == "Master’s degree"]['ConvertedCompYearly']
professional_salaries = df[df['EdLevel'] == "Professional degree"]['ConvertedCompYearly']

# Descriptive statistics for each group
print('Bachelor\'s Degree Salaries:')
print(bachelor_salaries.describe())

print('\nMaster\'s Degree Salaries:')
print(master_salaries.describe())

print('\nProfessional Degree Salaries:')
print(professional_salaries.describe())

In [ ]:
from scipy import stats

# Perform one-way ANOVA
anova_result = stats.f_oneway(bachelor_salaries, master_salaries, professional_salaries)

# Output the ANOVA results
print('ANOVA Result:')
print(f"F-statistic: {anova_result.statistic}")
print(f"P-value: {anova_result.pvalue}")


In [ ]:
# Number of bootstrap replications
n_bootstrap = 10000

# Function to perform bootstrap sampling
def bootstrap(data, n_samples=n_bootstrap):
    return np.random.choice(data, size=(n_samples, len(data)), replace=True)

# Bootstrap samples for each education group
bachelor_bootstrap = bootstrap(bachelor_salaries)
master_bootstrap = bootstrap(master_salaries)
professional_bootstrap = bootstrap(professional_salaries)

# Calculate the means of the bootstrap samples
bachelor_bootstrap_means = np.mean(bachelor_bootstrap, axis=1)
master_bootstrap_means = np.mean(master_bootstrap, axis=1)
professional_bootstrap_means = np.mean(professional_bootstrap, axis=1)

# Calculate the pairwise differences in means for each bootstrap sample
mean_diff_bachelor_master = bachelor_bootstrap_means - master_bootstrap_means
mean_diff_bachelor_professional = bachelor_bootstrap_means - professional_bootstrap_means
mean_diff_master_professional = master_bootstrap_means - professional_bootstrap_means

# 1. Plot the bootstrapped distributions of salaries for each group
plt.figure(figsize=(10, 6))
plt.hist(bachelor_bootstrap_means, bins=50, alpha=0.7, label="Bachelor's", color='blue')
plt.hist(master_bootstrap_means, bins=50, alpha=0.7, label="Master's", color='green')
plt.hist(professional_bootstrap_means, bins=50, alpha=0.7, label="Professional", color='orange')
plt.title('Bootstrapped Distributions of Salaries (Bachelor, Master, Professional)')
plt.xlabel('Salary')
plt.ylabel('Frequency')
plt.legend()
plt.show()

# 2. Plot the pairwise distribution of the differences in means
plt.figure(figsize=(10, 6))
plt.hist(mean_diff_bachelor_master, bins=50, alpha=0.7, label="Bachelor - Master", color='purple')
plt.hist(mean_diff_bachelor_professional, bins=50, alpha=0.7, label="Bachelor - Professional", color='red')
plt.hist(mean_diff_master_professional, bins=50, alpha=0.7, label="Master - Professional", color='brown')
plt.title('Distribution of the Difference in Mean Salaries (Pairwise)')
plt.xlabel('Difference in Salary')
plt.ylabel('Frequency')
plt.legend()
plt.show()

# 3. Normalize the differences in means (z-scores) and plot the normalized distribution
mean_diff_bachelor_master_z = (mean_diff_bachelor_master - np.mean(mean_diff_bachelor_master)) / np.std(mean_diff_bachelor_master)
mean_diff_bachelor_professional_z = (mean_diff_bachelor_professional - np.mean(mean_diff_bachelor_professional)) / np.std(mean_diff_bachelor_professional)
mean_diff_master_professional_z = (mean_diff_master_professional - np.mean(mean_diff_master_professional)) / np.std(mean_diff_master_professional)

plt.figure(figsize=(10, 6))
plt.hist(mean_diff_bachelor_master_z, bins=50, alpha=0.7, label="Bachelor - Master", color='purple')
plt.hist(mean_diff_bachelor_professional_z, bins=50, alpha=0.7, label="Bachelor - Professional", color='red')
plt.hist(mean_diff_master_professional_z, bins=50, alpha=0.7, label="Master - Professional", color='brown')
plt.title('Normalized Distribution of the Difference in Mean Salaries (Z-Scores)')
plt.xlabel('Z-score of Difference in Salary')
plt.ylabel('Frequency')
plt.legend()
plt.show()


In [ ]:
# Perform ANOVA on the bootstrapped data
anova_bootstrap_result = stats.f_oneway(bachelor_bootstrap_means, master_bootstrap_means, professional_bootstrap_means)

# Output the ANOVA results for the bootstrapped data
print('Bootstrap ANOVA Result:')
print(f"F-statistic (Bootstrap): {anova_bootstrap_result.statistic}")
print(f"P-value (Bootstrap): {anova_bootstrap_result.pvalue}")

# Compare with original ANOVA
print('\nComparison with Original ANOVA:')
print(f"F-statistic (Original): {anova_result.statistic}")
print(f"P-value (Original): {anova_result.pvalue}")
print(f"F-statistic (Bootstrap): {anova_bootstrap_result.statistic}")
print(f"P-value (Bootstrap): {anova_bootstrap_result.pvalue}")


## Conclusions

| Finding | Result |
|---|---|
| Remote vs in-person salary difference | ~$35,000 (p < 0.001) |
| ANOVA: education effect on salary | Significant (p < 0.05) |
| Assumptions satisfied? | No — normality and equal variance violated |
| Bootstrap confirms t-test? | Yes — gap is stable across 10,000 resamples |

**Key takeaways:**
- Remote workers earn ~92% more than in-person workers on average ($83k vs $43k)
- Professional degrees show the widest salary range; Bachelor's/Master's have similar medians
- Standard parametric tests are technically violated on this data, but bootstrap results align closely — the salary gaps are real and robust